# PyDI Data Integration Workflow: Music

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with music datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Loading and Profiling](#part-2-data-loading-and-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [1]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input" / "music"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "music"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

/usr/local/Caskroom/miniconda/base/envs/pydi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Target Schema and Normalization Spec

In [3]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

# Add list columns to spec
spec.set_column("tracks_track_name", output_type="list")
spec.set_column("tracks_track_duration", output_type="list")
spec.set_column("tracks_track_position", output_type="list")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,artist,string
3,release-date,string
4,release-country,string
5,duration,int
6,label,string
7,genre,string
8,tracks_track_name,list
9,tracks_track_duration,list


## Step 2: Load Source Datasets

In [4]:
from PyDI.io import load_xml, load_csv, load_json
mbrainz = load_xml(INPUT_DIR / "schemamatching" / "original_data" / "musicbrainz.xml")
mbrainz.attrs["dataset_name"] = "musicbrainz"
mbrainz.head()

,id,{http://musicbrainz.org/ns/mmd-2.0#}title,{http://musicbrainz.org/ns/mmd-2.0#}rel_id,{http://musicbrainz.org/ns/mmd-2.0#}status_id,{http://musicbrainz.org/ns/mmd-2.0#}status,{http://musicbrainz.org/ns/mmd-2.0#}quality,{http://musicbrainz.org/ns/mmd-2.0#}artist,{http://musicbrainz.org/ns/mmd-2.0#}date,{http://musicbrainz.org/ns/mmd-2.0#}country,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_count,...,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}position,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}number,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_id,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}title,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}length,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}video,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}artist-credit_{http://musicbrainz.org/ns/mmd-2.0#}name-credit_{http://musicbrainz.org/ns/mmd-2.0#}artist_id,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}artist-credit_{http://musicbrainz.org/ns/mmd-2.0#}name-credit_{http://musicbrainz.org/ns/mmd-2.0#}artist_{http://musicbrainz.org/ns/mmd-2.0#}name,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}data-track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}artist-credit_{http://musicbrainz.org/ns/mmd-2.0#}name-credit_{http://musicbrainz.org/ns/mmd-2.0#}artist_{http://musicbrainz.org/ns/mmd-2.0#}sort-name,{http://musicbrainz.org/ns/mmd-2.0#}medium-list_{http://musicbrainz.org/ns/mmd-2.0#}medium_{http://musicbrainz.org/ns/mmd-2.0#}track-list_{http://musicbrainz.org/ns/mmd-2.0#}track_{http://musicbrainz.org/ns/mmd-2.0#}recording_{http://musicbrainz.org/ns/mmd-2.0#}video
0,5d498d28-9001-4bd4-8507-a9a88cbd2f86,Fermats Theorem / Sight Beyond,mbrainz_1,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,John B,1996,GB,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,b155de7a-5964-4534-9d4e-923b73df141a,Tempest / Inner Sense,mbrainz_2,4e304316-386d-3409-af2e-78857eec5cfe,Official,high,Psychosis,1998-12-14,GB,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6387974a-d698-470f-ba01-beb9bc1d05a5,The Sign's Alive,mbrainz_3,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Lypid,2000-09-05,US,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,db0e7fa9-9e8e-4a97-81f0-b02c06b95

In [5]:
import re
# Clean column names from XML namespaces
def strip_all_ns(col):
    """
    Remove ALL {namespace} prefixes from a column name string.
    Example:
      '{ns}medium-list_{ns}medium_{ns}track' -> 'medium-list_medium_track'
    """
    if not isinstance(col, str):
        return col
    # remove every occurrence of {...}
    return re.sub(r"\{[^}]+\}", "", col)

mbrainz = mbrainz.rename(columns=strip_all_ns)
mbrainz.head()

,id,title,rel_id,status_id,status,quality,artist,date,country,medium-list_count,...,medium-list_medium_data-track-list_track_position,medium-list_medium_data-track-list_track_number,medium-list_medium_data-track-list_track_recording_id,medium-list_medium_data-track-list_track_recording_title,medium-list_medium_data-track-list_track_recording_length,medium-list_medium_data-track-list_track_recording_video,medium-list_medium_data-track-list_track_recording_artist-credit_name-credit_artist_id,medium-list_medium_data-track-list_track_recording_artist-credit_name-credit_artist_name,medium-list_medium_data-track-list_track_recording_artist-credit_name-credit_artist_sort-name,medium-list_medium_track-list_track_recording_video
0,5d498d28-9001-4bd4-8507-a9a88cbd2f86,Fermats Theorem / Sight Beyond,mbrainz_1,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,John B,1996,GB,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,b155de7a-5964-4534-9d4e-923b73df141a,Tempest / Inner Sense,mbrainz_2,4e304316-386d-3409-af2e-78857eec5cfe,Official,high,Psychosis,1998-12-14,GB,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6387974a-d698-470f-ba01-beb9bc1d05a5,The Sign's Alive,mbrainz_3,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Lypid,2000-09-05,US,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,db0e7fa9-9e8e-4a97-81f0-b02c06b95a4f,Surrender,mbrainz_4,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Petalpusher,1999-04-27,US,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,503cb223-719b-332f-bd81-8d3e182a0308,Unreasonable Behaviour,mbrainz_6,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,"Garnier, Laurent",2000-07-24,FR,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# find percentage of NaN values in each column
nan_percentage = mbrainz.isna().mean() * 100
print(nan_percentage)

# drop columns with NaN percentage over 80%
mbrainz = mbrainz.loc[:, nan_percentage <= 50]
print(mbrainz.columns.tolist())

mbrainz.head()


id                                                                                                0.000000
title                                                                                             0.000000
rel_id                                                                                            0.000000
status_id                                                                                         2.120512
status                                                                                            2.120512
quality                                                                                           0.000000
artist                                                                                            0.000000
date                                                                                              6.571489
country                                                                                           9.657779
medium-list_count                    

,id,title,rel_id,status_id,status,quality,artist,date,country,medium-list_count,...,medium-list_medium_track-list_track_id,medium-list_medium_track-list_track_position,medium-list_medium_track-list_track_number,medium-list_medium_track-list_track_length,medium-list_medium_track-list_track_recording_id,medium-list_medium_track-list_track_recording_title,medium-list_medium_track-list_track_recording_length,medium-list_medium_track-list_track_recording_artist-credit_name-credit_artist_id,medium-list_medium_track-list_track_recording_artist-credit_name-credit_artist_name,medium-list_medium_track-list_track_recording_artist-credit_name-credit_artist_sort-name
0,5d498d28-9001-4bd4-8507-a9a88cbd2f86,Fermats Theorem / Sight Beyond,mbrainz_1,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,John B,1996,GB,1,...,"[7acb267d-e145-369d-be1c-91cfdb55cb81, 403abcc...","[1, 2]","[A, B]","[558000, 497000]","[bab13509-c16e-4322-b241-9bdd60a54256, c69b4ae...","[Fermats Theorem, Sight Beyond]","[558000, 497000]","[353856b7-8d79-4136-9ac6-7e47954e5be9, 353856b...","[John B, John B]","[John B, John B]"
1,b155de7a-5964-4534-9d4e-923b73df141a,Tempest / Inner Sense,mbrainz_2,4e304316-386d-3409-af2e-78857eec5cfe,Official,high,Psychosis,1998-12-14,GB,1,...,"[3bca38d7-8946-38cd-be5b-323036b18db3, ac71ecb...","[1, 2]","[A, B]","[364000, 360000]","[10350648-79c3-4c01-91b0-35ae681a1187, 455f00e...","[Tempest, Inner Sense]","[364000, 360000]","[19b62aba-6456-441c-9de4-2ce34f199369, 19b62ab...","[Psychosis, Psychosis]","[Psychosis, Psychosis]"
2,6387974a-d698-470f-ba01-beb9bc1d05a5,The Sign's Alive,mbrainz_3,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Lypid,2000-09-05,US,1,...,"[06f29346-c30f-3e72-b109-fa3d52bd5dd9, 8d40a89...","[1, 2, 3, 4, 5, 6]","[1, 2, 3, 4, 5, 6]","[366000, 284000, 331000, 331000, 655000, 417000]","[0084b37c-300d-4f07-9a34-450e49b43ac6, db2045f...","[The Sign's Alive (original mix), The Sign's A...","[366000, 284000, 331000, 331000, 655000, 417000]","[2244c07d-db3d-40b3-9d47-cea23ee6fe4d, 2244c07...","[Lypid, Lypid, Lypid, Lypid, Lypid, Lypid]","[Lypid, Lypid, Lypid, Lypid, Lypid, Lypid]"
3,db0e7fa9-9e8e-4a97-81f0-b02c06b95a4f,Surrender,mbrainz_4,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Petalpusher,1999-04-27,US,1,...,"[30f5eeb1-6370-3e42-b5fe-5e9dbe1df7e6, a94b06e...","[1, 2, 3, 4]","[A1, A2, B1, B2]","[405000, 411000, 420000, 390000]","[853197b8-0b89-40d6-97c5-093643f178b6, f70dff2...","[Surrender (Petalpusher original), Surrender (...","[405000, 411000, 420000, 390000]","[4f5615f4-3845-4173-a162-ba155af63ad4, 4f5615f...","[Petalpusher, Petalpusher, Petalpusher, Petalp...","[Petalpusher, Petalpusher, Petalpusher, Petalp..."
4,503cb223-719b-332f-bd81-8d3e182a0308,Unreasonable Behaviour,mbrainz_6,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,"Garnier, Laurent",2000-07-24,FR,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
lastfm = load_csv(INPUT_DIR / "schemamatching" / "original_data" / "lastfm.csv")
lastfm.attrs["dataset_name"] = "lastfm"
lastfm.head()

,artist,name,url,id,listeners,playcount,tracks_track_name,tracks_track_rank,tracks_track_duration,tracks_track_artist
0,John B,John B - Fermats Theorem / Sight Beyond,http://www.last.fm/music/John+B/Fermats+Theore...,lastFM_1,7.0,27.0,"['Fermats Theorem', 'Sight Beyond']","['1', '2']","['406', '497']","['John B', 'John B']"
1,Psychosis,Tempest / Inner Sense,http://www.last.fm/music/Psychosis/Tempest+%2F...,lastFM_2,23.0,46.0,"['Tempest', 'Inner Sense']","['1', '2']","['371', '363']","['Psychosis', 'Psychosis']"
2,Petalpusher,Petalpusher - Surrender,http://www.last.fm/music/Petalpusher/Surrender,lastFM_4,61.0,281.0,"['Surrender (Petalpusher Original)', ""Surrende...","['1', '2', '3', '4']","['405', '411', '420', '390']","['Petalpusher', 'Petalpusher', 'Petalpusher', ..."
3,R. Trent,in the spirit,http://www.last.fm/music/Ron+Trent/in+the+spirit,lastFM_8,18.0,53.0,"['In The Spirit (The Full Experience)', 'In Th...","['1', '2', '3']","['467', '329', '469']","['Ron Trent', 'Ron Trent', 'Ron Trent']"
4,S. Vitus Dance,Come Of Age,http://www.last.fm/music/St.+Vitus+Dance/Come+...,lastFM_11,21.0,137.0,"['Bliss', 'Tunnel Vision', 'Catch The Sun', 'M...","['1', '2', '3', '4']","['345', '409', '333', '291']","['St. Vitus Dance', 'St. Vitus Dance', 'St. Vi..."


In [8]:
discogs = load_json(INPUT_DIR / "schemamatching" / "original_data" / "discogs.json")
discogs.attrs["dataset_name"] = "discogs"
discogs.head()

,artist,name,id,country,genre,label,released,tracks_track_name,tracks_track_rank,tracks_track_duration
0,John B,Fermats Theorem / Sight Beyond,discogs_3,UK,Electronic,New Identity Recordings,1996-00-00,"[Fermats Theorem, Sight Beyond]","[1, 2]",None
1,Psychosis,Tempest / Inner Sense,discogs_4,UK,Electronic,Renegade Hardware,1998-00-00,"[Tempest, Inner Sense]","[1, 2]",None
2,Lypid,The Sign's Alive,discogs_5,US,Electronic,Statra Recordings,2000-09-05,"[The Sign's Alive (Original Mix), The Sign's A...","[1, 2, 3, 4, 5, 6]",None
3,Petalpusher,Surrender,discogs_6,US,Electronic,Naked Music Recordings,1999-04-27,"[Surrender (Petalpusher Original), Surrender (...","[1, 2, 3, 4]","[6:45, 6:51, 7:00, 6:30]"
4,Laurent Garnier,Unreasonable Behaviour,discogs_11,France,Electronic,F Communications,2000-06-00,"[The Warning, City Sphere, Forgotten Thoughts,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1:48, 6:15, 6:48, 7:10, 1:20, 5:00, 9:10, 4:5..."


## Step 3: LLM-Based Schema Matching

In [9]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match mbrainz dataset
mbrainz_mapping = matcher.match(mbrainz, df_target)

mbrainz_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,musicbrainz,id,target_schema,id,0.95,llm_based_matching
1,musicbrainz,title,target_schema,name,0.95,llm_based_matching
2,musicbrainz,artist,target_schema,artist,0.95,llm_based_matching
3,musicbrainz,date,target_schema,release-date,0.95,llm_based_matching
4,musicbrainz,country,target_schema,release-country,0.95,llm_based_matching
5,musicbrainz,medium-list_medium_track-list_track_position,target_schema,tracks_track_position,0.95,llm_based_matching
6,musicbrainz,medium-list_medium_track-list_track_length,target_schema,tracks_track_duration,0.95,llm_based_matching
7,musicbrainz,medium-list_medium_track-list_track_recording_...,target_schema,tracks_track_name,0.95,llm_based_matching


In [10]:
lastfm_mapping = matcher.match(lastfm, df_target)
lastfm_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,lastfm,artist,target_schema,artist,0.95,llm_based_matching
1,lastfm,name,target_schema,name,0.95,llm_based_matching
2,lastfm,id,target_schema,id,0.95,llm_based_matching
3,lastfm,tracks_track_name,target_schema,tracks_track_name,0.95,llm_based_matching
4,lastfm,tracks_track_rank,target_schema,tracks_track_position,0.95,llm_based_matching
5,lastfm,tracks_track_duration,target_schema,tracks_track_duration,0.95,llm_based_matching


In [11]:
discogs_mapping = matcher.match(discogs, df_target)
discogs_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,discogs,artist,target_schema,artist,0.95,llm_based_matching
1,discogs,name,target_schema,name,0.95,llm_based_matching
2,discogs,id,target_schema,id,0.95,llm_based_matching
3,discogs,country,target_schema,release-country,0.95,llm_based_matching
4,discogs,genre,target_schema,genre,0.95,llm_based_matching
5,discogs,label,target_schema,label,0.95,llm_based_matching
6,discogs,released,target_schema,release-date,0.95,llm_based_matching
7,discogs,tracks_track_name,target_schema,tracks_track_name,0.95,llm_based_matching
8,discogs,tracks_track_rank,target_schema,tracks_track_position,0.95,llm_based_matching
9,discogs,tracks_track_duration,target_schema,tracks_track_duration,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [12]:
# convert lastfm string list representations into actual lists
import ast
import numpy as np

def parse_to_int_list_or_nan(x):
    # 1. Missing values → NaN
    if pd.isna(x):
        return np.nan

    # Helper: convert elements to integers if possible
    def convert_numeric_list(lst):
        new = []
        for item in lst:
            # Convert strings like "406" or "30" to int
            if isinstance(item, str) and item.isdigit():
                new.append(int(item))
            # Already an integer
            elif isinstance(item, int):
                new.append(item)
            else:
                # leave non-numeric as-is
                new.append(item)
        return new

    # 2. Already a Python list → enforce numeric conversion
    if isinstance(x, list):
        return convert_numeric_list(x)

    # 3. Strings: attempt literal_eval
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return np.nan
        
        try:
            value = ast.literal_eval(s)
        except (ValueError, SyntaxError):
            # Not a literal → wrap as single-element list
            return [s]

        # Case A: literal became a list
        if isinstance(value, list):
            return convert_numeric_list(value)

        # Case B: literal became a scalar (int, str, etc.)
        # Wrap into single-element list
        return convert_numeric_list([value])

    # 4. Any other non-list scalar → wrap and convert
    return convert_numeric_list([x])


list_columns = ["tracks_track_rank", "tracks_track_name", "tracks_track_duration", "tracks_track_artist"]

for col in list_columns:
    lastfm[col] = lastfm[col].apply(parse_to_int_list_or_nan)

lastfm.head()

,artist,name,url,id,listeners,playcount,tracks_track_name,tracks_track_rank,tracks_track_duration,tracks_track_artist
0,John B,John B - Fermats Theorem / Sight Beyond,http://www.last.fm/music/John+B/Fermats+Theore...,lastFM_1,7.0,27.0,"[Fermats Theorem, Sight Beyond]","[1, 2]","[406, 497]","[John B, John B]"
1,Psychosis,Tempest / Inner Sense,http://www.last.fm/music/Psychosis/Tempest+%2F...,lastFM_2,23.0,46.0,"[Tempest, Inner Sense]","[1, 2]","[371, 363]","[Psychosis, Psychosis]"
2,Petalpusher,Petalpusher - Surrender,http://www.last.fm/music/Petalpusher/Surrender,lastFM_4,61.0,281.0,"[Surrender (Petalpusher Original), Surrender (...","[1, 2, 3, 4]","[405, 411, 420, 390]","[Petalpusher, Petalpusher, Petalpusher, Petalp..."
3,R. Trent,in the spirit,http://www.last.fm/music/Ron+Trent/in+the+spirit,lastFM_8,18.0,53.0,"[In The Spirit (The Full Experience), In The S...","[1, 2, 3]","[467, 329, 469]","[Ron Trent, Ron Trent, Ron Trent]"
4,S. Vitus Dance,Come Of Age,http://www.last.fm/music/St.+Vitus+Dance/Come+...,lastFM_11,21.0,137.0,"[Bliss, Tunnel Vision, Catch The Sun, Mystic V...","[1, 2, 3, 4]","[345, 409, 333, 291]","[St. Vitus Dance, St. Vitus Dance, St. Vitus D..."


In [13]:
# handle mbrainz track duration (milliseconds to seconds)
def ms_to_seconds(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return np.nan

    if isinstance(x, list):
        return [ms_to_seconds(v) for v in x]

    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x // 1000)

    if isinstance(x, str) and x.strip().isdigit():
        return int(int(x.strip()) // 1000)

    return np.nan

mbrainz["medium-list_medium_track-list_track_length"] = mbrainz["medium-list_medium_track-list_track_length"].apply(ms_to_seconds)
mbrainz.head()

,id,title,rel_id,status_id,status,quality,artist,date,country,medium-list_count,...,medium-list_medium_track-list_track_id,medium-list_medium_track-list_track_position,medium-list_medium_track-list_track_number,medium-list_medium_track-list_track_length,medium-list_medium_track-list_track_recording_id,medium-list_medium_track-list_track_recording_title,medium-list_medium_track-list_track_recording_length,medium-list_medium_track-list_track_recording_artist-credit_name-credit_artist_id,medium-list_medium_track-list_track_recording_artist-credit_name-credit_artist_name,medium-list_medium_track-list_track_recording_artist-credit_name-credit_artist_sort-name
0,5d498d28-9001-4bd4-8507-a9a88cbd2f86,Fermats Theorem / Sight Beyond,mbrainz_1,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,John B,1996,GB,1,...,"[7acb267d-e145-369d-be1c-91cfdb55cb81, 403abcc...","[1, 2]","[A, B]","[558, 497]","[bab13509-c16e-4322-b241-9bdd60a54256, c69b4ae...","[Fermats Theorem, Sight Beyond]","[558000, 497000]","[353856b7-8d79-4136-9ac6-7e47954e5be9, 353856b...","[John B, John B]","[John B, John B]"
1,b155de7a-5964-4534-9d4e-923b73df141a,Tempest / Inner Sense,mbrainz_2,4e304316-386d-3409-af2e-78857eec5cfe,Official,high,Psychosis,1998-12-14,GB,1,...,"[3bca38d7-8946-38cd-be5b-323036b18db3, ac71ecb...","[1, 2]","[A, B]","[364, 360]","[10350648-79c3-4c01-91b0-35ae681a1187, 455f00e...","[Tempest, Inner Sense]","[364000, 360000]","[19b62aba-6456-441c-9de4-2ce34f199369, 19b62ab...","[Psychosis, Psychosis]","[Psychosis, Psychosis]"
2,6387974a-d698-470f-ba01-beb9bc1d05a5,The Sign's Alive,mbrainz_3,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Lypid,2000-09-05,US,1,...,"[06f29346-c30f-3e72-b109-fa3d52bd5dd9, 8d40a89...","[1, 2, 3, 4, 5, 6]","[1, 2, 3, 4, 5, 6]","[366, 284, 331, 331, 655, 417]","[0084b37c-300d-4f07-9a34-450e49b43ac6, db2045f...","[The Sign's Alive (original mix), The Sign's A...","[366000, 284000, 331000, 331000, 655000, 417000]","[2244c07d-db3d-40b3-9d47-cea23ee6fe4d, 2244c07...","[Lypid, Lypid, Lypid, Lypid, Lypid, Lypid]","[Lypid, Lypid, Lypid, Lypid, Lypid, Lypid]"
3,db0e7fa9-9e8e-4a97-81f0-b02c06b95a4f,Surrender,mbrainz_4,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,Petalpusher,1999-04-27,US,1,...,"[30f5eeb1-6370-3e42-b5fe-5e9dbe1df7e6, a94b06e...","[1, 2, 3, 4]","[A1, A2, B1, B2]","[405, 411, 420, 390]","[853197b8-0b89-40d6-97c5-093643f178b6, f70dff2...","[Surrender (Petalpusher original), Surrender (...","[405000, 411000, 420000, 390000]","[4f5615f4-3845-4173-a162-ba155af63ad4, 4f5615f...","[Petalpusher, Petalpusher, Petalpusher, Petalp...","[Petalpusher, Petalpusher, Petalpusher, Petalp..."
4,503cb223-719b-332f-bd81-8d3e182a0308,Unreasonable Behaviour,mbrainz_6,4e304316-386d-3409-af2e-78857eec5cfe,Official,normal,"Garnier, Laurent",2000-07-24,FR,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# handle discogs track duration format "MM:SS"
def parse_duration_to_seconds(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return np.nan    
    if isinstance(x, list):
        return [parse_duration_to_seconds(v) for v in x]
    if isinstance(x, str):
        s = x.strip()
        if ":" in s:
            parts = s.split(":")
            if len(parts) == 2:
                try:
                    return int(parts[0]) * 60 + int(parts[1])
                except ValueError:
                    return np.nan
        return np.nan
    return np.nan

discogs["tracks_track_duration"] = discogs["tracks_track_duration"].apply(parse_duration_to_seconds)
discogs.head()

,artist,name,id,country,genre,label,released,tracks_track_name,tracks_track_rank,tracks_track_duration
0,John B,Fermats Theorem / Sight Beyond,discogs_3,UK,Electronic,New Identity Recordings,1996-00-00,"[Fermats Theorem, Sight Beyond]","[1, 2]",NaN
1,Psychosis,Tempest / Inner Sense,discogs_4,UK,Electronic,Renegade Hardware,1998-00-00,"[Tempest, Inner Sense]","[1, 2]",NaN
2,Lypid,The Sign's Alive,discogs_5,US,Electronic,Statra Recordings,2000-09-05,"[The Sign's Alive (Original Mix), The Sign's A...","[1, 2, 3, 4, 5, 6]",NaN
3,Petalpusher,Surrender,discogs_6,US,Electronic,Naked Music Recordings,1999-04-27,"[Surrender (Petalpusher Original), Surrender (...","[1, 2, 3, 4]","[405, 411, 420, 390]"
4,Laurent Garnier,Unreasonable Behaviour,discogs_11,France,Electronic,F Communications,2000-06-00,"[The Warning, City Sphere, Forgotten Thoughts,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[108, 375, 408, 430, 80, 300, 550, 294, 408, 5..."


In [15]:
# Handle 00 days and months in discogs
discogs["released"] = discogs["released"].apply(lambda x: re.sub(r"-00", "-01", x) if isinstance(x, str) else x)
discogs.iloc[0]["released"]

'1996-01-01'

In [16]:
# replace id with rel_id in mbrainz
mbrainz = mbrainz.drop(columns=["id"])
mbrainz = mbrainz.rename(columns={"rel_id": "id"})

In [17]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("release-country", country_format="name")
spec.set_column("release-date", output_type="datetime")

discogs_normalized = translator.translate(
    discogs, discogs_mapping,
    normalize =spec, on_failure="keep"
)

lastfm_normalized = translator.translate(
    lastfm, lastfm_mapping,
    normalize=spec, on_failure="keep"
)

mbrainz_normalized = translator.translate(
    mbrainz, mbrainz_mapping,
    normalize=spec, on_failure="keep"
)

In [18]:
# Inspect normalized mbrainz dataset (target columns only)
mbrainz_cols = [c for c in target_columns if c in mbrainz_normalized.columns]
mbrainz_normalized[mbrainz_cols].head(10)

,id,name,artist,release-date,release-country,tracks_track_name,tracks_track_duration,tracks_track_position
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom,"[Fermats Theorem, Sight Beyond]","[558, 497]","[1, 2]"
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom,"[Tempest, Inner Sense]","[364, 360]","[1, 2]"
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States,"[The Sign's Alive (original mix), The Sign's A...","[366, 284, 331, 331, 655, 417]","[1, 2, 3, 4, 5, 6]"
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States,"[Surrender (Petalpusher original), Surrender (...","[405, 411, 420, 390]","[1, 2, 3, 4]"
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,NaN,NaN,NaN
5,mbrainz_7,In the Spirit,"Trent, Ron",1999-01-01,United Kingdom,"[In the Spirit (The Full Experience), In the S...","[477, 343, 471]","[1, 2, 3]"
6,mbrainz_8,The Dance,"Gallery Collective, The",1996-01-01,United States,"[The Dance (The Full Gallery mix), The Dance (...","[794, 532, 543]","[1, 2, 3]"
7,mbrainz_9,Come of Age,St. Vitus Dance,1994-01-01,United Kingdom,"[Bliss, Tunnel Vision, Catch the Sun, Mystic V...","[347, 411, 335, 370]","[1, 2, 3, 4]"
8,mbrainz_10,Contrax / All Mighty,Decorum,1999-01-01,United Kingdom,"[Contrax, All Mighty]","[417, 362]","[1, 2]"
9,mbrainz_11,Electronically Tested,Surgeon,1995-01-01,United Kingdom,"[Barrier Method, Pork Machine, Language Barrie...","[333, 404, 458, 418]","[1, 2, 3, 4]"


In [19]:
discogs_cols = [c for c in target_columns if c in discogs_normalized.columns]
discogs_normalized[discogs_cols].head(10)

,id,name,artist,release-date,release-country,label,genre,tracks_track_name,tracks_track_duration,tracks_track_position
0,discogs_3,Fermats Theorem / Sight Beyond,John B,1996-01-01,Uganda,New Identity Recordings,Electronic,"[Fermats Theorem, Sight Beyond]",NaN,"[1, 2]"
1,discogs_4,Tempest / Inner Sense,Psychosis,1998-01-01,Uganda,Renegade Hardware,Electronic,"[Tempest, Inner Sense]",NaN,"[1, 2]"
2,discogs_5,The Sign's Alive,Lypid,2000-09-05,United States,Statra Recordings,Electronic,"[The Sign's Alive (Original Mix), The Sign's A...",NaN,"[1, 2, 3, 4, 5, 6]"
3,discogs_6,Surrender,Petalpusher,1999-04-27,United States,Naked Music Recordings,Electronic,"[Surrender (Petalpusher Original), Surrender (...","[405, 411, 420, 390]","[1, 2, 3, 4]"
4,discogs_11,Unreasonable Behaviour,Laurent Garnier,2000-06-01,France,F Communications,Electronic,"[The Warning, City Sphere, Forgotten Thoughts,...","[108, 375, 408, 430, 80, 300, 550, 294, 408, 5...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
5,discogs_13,In The Spirit,Ron Trent,1999-01-01,Uganda,Peacefrog Records,Electronic,"[In The Spirit (The Full Experience), In The S...",NaN,"[1, 2, 3]"
6,discogs_14,The Dance,The Gallery Collective,1996-01-01,United States,Prescription,Electronic,"[The Dance (The Full Gallery Mix), The Dance (...",NaN,"[1, 2, 3]"
7,discogs_16,Analogue,Mampi Swift,1997-01-01,Uganda,Suburban Base Records,Electronic,"[Analogue, Behold]",NaN,"[1, 2]"
8,discogs_17,Come Of Age,St. Vitus Dance,1994-01-01,Uganda,Peacefrog Records,Electronic,"[Bliss, Tunnel Vision, Catch The Sun, Mystic V...",NaN,"[1, 2, 3, 4]"
9,discogs_18,Ebony Angel - The Resurrection,Monica Elam,1999-01-01,United States,Clairaudience,Electronic,"[The Resurrection (Rainforest Rhapsody), The R...",NaN,"[1, 2, 3, 4]"


In [20]:
lastfm_cols = [c for c in target_columns if c in lastfm_normalized.columns]
lastfm_normalized[lastfm_cols].head(10)

,id,name,artist,tracks_track_name,tracks_track_duration,tracks_track_position
0,lastFM_1,John B - Fermats Theorem / Sight Beyond,John B,"[Fermats Theorem, Sight Beyond]","[406, 497]","[1, 2]"
1,lastFM_2,Tempest / Inner Sense,Psychosis,"[Tempest, Inner Sense]","[371, 363]","[1, 2]"
2,lastFM_4,Petalpusher - Surrender,Petalpusher,"[Surrender (Petalpusher Original), Surrender (...","[405, 411, 420, 390]","[1, 2, 3, 4]"
3,lastFM_8,in the spirit,R. Trent,"[In The Spirit (The Full Experience), In The S...","[467, 329, 469]","[1, 2, 3]"
4,lastFM_11,Come Of Age,S. Vitus Dance,"[Bliss, Tunnel Vision, Catch The Sun, Mystic V...","[345, 409, 333, 291]","[1, 2, 3, 4]"
5,lastFM_16,Devotional,D. Alvarado,"[Devotional, Sunstone]","[466, 478]","[1, 2]"
6,lastFM_24,P. Johnson - The Music In Me,P. Johnson,"[The Music In Me, Slinky, Y'All Stole Them Dan...","[352, 186, 229, 237, 164]","[1, 2, 3, 4, 5]"
7,lastFM_27,On and,GH-106,"[XJ6, Fantasy]","[90, 30]","[1, 2]"
8,lastFM_28,- new - Subtle Frequencies,C. Jackson,"[Check our Beats, Teleport]","[307, 306]","[1, 2]"
9,lastFM_30,Second Area / Think Tank,Inigo Kennedy,"[Second Area, Think Tank]","[439, 388]","[1, 2]"


In [21]:
# Only keep target columns
mbrainz = mbrainz_normalized[mbrainz_cols].copy()
lastfm = lastfm_normalized[lastfm_cols].copy()
discogs = discogs_normalized[discogs_cols].copy()

In [22]:
# Sum up track duration lists into total album duration
def sum_track_durations(durations):
    if isinstance(durations, list):
        vals = [int(d) for d in durations if isinstance(d, (int, float, np.integer, np.floating))]
        return sum(vals) if vals else pd.NA
    return pd.NA

# for discogs and mbrainz put scalars within list attributes into lists
def wrap_in_list_if_scalar(x):
    if isinstance(x, list):
        return x
    elif pd.isna(x):
        return x
    else:
        return [x]

for col in ["tracks_track_name", "tracks_track_duration", "tracks_track_position"]:
    discogs[col] = discogs[col].apply(wrap_in_list_if_scalar)
    mbrainz[col] = mbrainz[col].apply(wrap_in_list_if_scalar)

lastfm["duration"] = lastfm["tracks_track_duration"].apply(sum_track_durations)
discogs["duration"] = discogs["tracks_track_duration"].apply(sum_track_durations)
mbrainz["duration"] = mbrainz["tracks_track_duration"].apply(sum_track_durations)

## Part 2: Data Loading and Profiling

In [23]:
# Display basic information
datasets = [discogs, mbrainz, lastfm]
names = ["Discogs", "MusicBrainz", "Last.fm"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 37,255


In [24]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

discogs:
  Rows: 22,627
  Columns: 11
  Total nulls: 22,300
  Null percentage: 9.0%
  Null counts per column:
    release-date: 2,234 (9.9%)
    release-country: 600 (2.7%)
    tracks_track_duration: 9,733 (43.0%)
    duration: 9,733 (43.0%)

musicbrainz:
  Rows: 4,763
  Columns: 9
  Total nulls: 2,815
  Null percentage: 6.6%
  Null counts per column:
    release-date: 313 (6.6%)
    release-country: 460 (9.7%)
    tracks_track_name: 256 (5.4%)
    tracks_track_duration: 765 (16.1%)
    tracks_track_position: 256 (5.4%)
    duration: 765 (16.1%)

lastfm:
  Rows: 9,865
  Columns: 7
  Total nulls: 21,052
  Null percentage: 30.5%
  Null counts per column:
    tracks_track_name: 5,263 (53.4%)
    tracks_track_duration: 5,263 (53.4%)
    tracks_track_position: 5,263 (53.4%)
    duration: 5,263 (53.4%)



{'rows': 9865,
 'columns': 7,
 'nulls_total': 21052,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'artist': 0,
  'tracks_track_name': 5263,
  'tracks_track_duration': 5263,
  'tracks_track_position': 5263,
  'duration': 5263},
 'dtypes': {'id': 'object',
  'name': 'object',
  'artist': 'object',
  'tracks_track_name': 'object',
  'tracks_track_duration': 'object',
  'tracks_track_position': 'object',
  'duration': 'object'}}

### Attribute Coverage Analysis

In [25]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,discogs_count,discogs_pct,discogs_coverage,discogs_samples,musicbrainz_count,musicbrainz_pct,musicbrainz_coverage,musicbrainz_samples,lastfm_count,lastfm_pct,lastfm_coverage,lastfm_samples,avg_coverage,max_coverage,datasets_with_attribute
0,artist,22627/22627,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",4763/4763,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",9865/9865,100.0%,1.000000,"['John B', 'Psychosis', 'Petalpusher']",1.000000,1.000000,3
1,duration,12894/22627,57.0%,0.569850,"[1626, 5938, 1008]",3998/4763,83.9%,0.839387,"[1055, 724, 2384]",4602/9865,46.6%,0.466498,"[903, 734, 1626]",0.625245,0.839387,3
2,genre,22627/22627,100.0%,1.000000,"['Electronic', 'Electronic', 'Electronic']",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
3,id,22627/22627,100.0%,1.000000,"['discogs_3', 'discogs_4', 'discogs_5']",4763/4763,100.0%,1.000000,"['mbrainz_1', 'mbrainz_2', 'mbrainz_3']",9865/9865,100.0%,1.000000,"['lastFM_1', 'lastFM_2', 'lastFM_4']",1.000000,1.000000,3
4,label,22627/22627,100.0%,1.000000,"['New Identity Recordings', 'Renegade Hardware...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
5,name,22627/22627,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",4763/4763,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",9865/9865,100.0%,1.000000,"['John B - Fermats Theorem / Sight Beyond', '...",1.000000,1.000000,3
6,release-country,22027/22627,97.3%,0.973483,"['Uganda', 'Uganda', 'United States']",4303/4763,90.3%,0.903422,"['United Kingdom', 'United Kingdom', 'United S...",0/0,0%,0.000000,N/A,0.625635,0.973483,2
7,release-date,20393/22627,90.1%,0.901268,"[Timestamp('1996-01-01 00:00:00'), Timestamp('...",4450/4763,93.4%,0.934285,"[Timestamp('1996-01-01 00:00:00'), Timestamp('...",0/0,0%,0.000000,N/A,0.611851,0.934285,2
8,tracks_track_duration,12894/22627,57.0%,0.569850,"[[405, 411, 420, 390], [108, 375, 408, 430, 80...",3998/4763,83.9%,0.839387,"[[558, 497], [364, 360], [366, 284, 331, 331, ...",4602/9865,46.6%,0.466498,"[[406, 497], [371, 363], [405, 411, 420, 390]]",0.625245,0.839387,3
9,tracks_track_name,22627/22627,100.0%,1.000000,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",4507/4763,94.6%,0.946252,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",4602/9865,46.6%,0.466498,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",0.804250,1.000000,3



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['artist', 'duration', 'id', 'name', 'release-country', 'release-date', 'tracks_track_duration', 'tracks_track_name', 'tracks_track_position']


### Detailed Data Profiling

In [26]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling Discogs...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 154.82it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles/discogs_profile.html
Profiling MusicBrainz...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 268.95it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles/musicbrainz_profile.html
Profiling Last.fm...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 267.96it/s]

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles/lastfm_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • discogs_profile.html
  • musicbrainz_profile.html
  • lastfm_profile.html


## Part 3: Entity Matching

### Step 1: Blocking

In [27]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [28]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - Longest Token in Name
# Add name_longest_token directly to the original dataframes
def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

mbrainz['name_longest_token'] = mbrainz['name'].apply(get_longest_token)
discogs['name_longest_token'] = discogs['name'].apply(get_longest_token)
lastfm['name_longest_token'] = lastfm['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    mbrainz, discogs,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

standard_blocker_m2l = StandardBlocker(
    mbrainz, lastfm,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4685 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1443 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2850 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1368 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/m

### Step 2: Evaluate Blocking Against Ground Truth

In [29]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 10 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 13 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 25 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 42 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 48 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 55 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 60 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 63 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 79 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 98 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 110 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 131 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 137 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 141 true matches
[INFO ]

{'pair_completeness': 1.0,
 'pair_quality': 0.002145844289888658,
 'reduction_ratio': 0.9967007230357613,
 'total_candidates': 355571,
 'total_possible_pairs': 107772401,
 'true_positives_found': 763,
 'total_true_pairs': 763,
 'batches_processed': 356,
 'evaluation_timestamp': '2025-12-13T10:57:54.972064',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/blocking_detailed_results.csv']}

In [30]:
# Now evaluate blocking for musicbrainz and lastfm combination

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2l,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 17 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 40 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 51 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 65 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 93 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 114 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 143 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 181 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 202 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 238 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 269 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 337 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 377 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 406 true matches
[I

{'pair_completeness': 0.9292682926829269,
 'pair_quality': 0.004941345835846157,
 'reduction_ratio': 0.9967180493240736,
 'total_candidates': 154209,
 'total_possible_pairs': 46986995,
 'true_positives_found': 762,
 'total_true_pairs': 820,
 'batches_processed': 155,
 'evaluation_timestamp': '2025-12-13T10:58:21.949177',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [31]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release duration — within 10% --> allow 10% deviation
    NumericComparator(
        column='duration',
        method='relative_difference',
        max_difference=0.10
    ),
    # # Track list — overlap
    StringComparator(
        column='tracks_track_name',
        similarity_function='jaccard',
        preprocess=normalize_text,
        list_strategy="set_overlap"
    ),
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [32]:
import numpy as np

# Convert lists in mbrainz["duration"] to single integer values (sum if list, else int)

def sum_duration(val):
    if isinstance(val, list):
        return int(np.nansum([int(x) for x in val if str(x).isdigit()]))
    try:
        return int(val)
    except Exception:
        return np.nan

mbrainz["duration"] = mbrainz["duration"].apply(sum_duration)

In [33]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=mbrainz,
    df_right=discogs, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.5,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 22627 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 22627 elements after 0:00:0.236; 355571 blocked pairs (reduction ratio: 0.9967007230357613)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:201.372; found 3722 correspondences.


In [34]:
matcher = RuleBasedMatcher()

# Remove release-date and release-country comparator since those columns are not present in lastfm dataset
comparators.pop()
comparators.pop()

correspondences_m2l = matcher.match(
    df_left=mbrainz,
    df_right=lastfm, 
    candidates=standard_blocker_m2l,
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.3,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 9865 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 9865 elements after 0:00:0.117; 154209 blocked pairs (reduction ratio: 0.9967180493240736)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:64.985; found 4428 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [35]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  471
[INFO ] root -   True Negatives:  5158
[INFO ] root -   False Positives: 172
[INFO ] root -   False Negatives: 4
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.970
[INFO ] root -   Precision: 0.733
[INFO ] root -   Recall:    0.992
[INFO ] root -   F1-Score:  0.843


{'precision': 0.7325038880248833,
 'recall': 0.991578947368421,
 'f1': 0.8425760286225402,
 'accuracy': 0.9696813092161929,
 'true_positives': 471,
 'false_positives': 172,
 'false_negatives': 4,
 'true_negatives': 5158,
 'threshold_used': 0.0,
 'total_correspondences': 3722,
 'filtered_correspondences': 3722,
 'evaluation_timestamp': '2025-12-13T11:02:56.941568',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_detailed_results.csv']}

In [36]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2956 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2486	|	84.10%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	317	|	10.72%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	93	|	3.15%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	28	|	0.95%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	18	|	0.61%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	7	|	0.24%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	3	|	0.10%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	2	|	0.07%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	1	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.03%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/music/cluster_analysis/cluster_size_distribution.csv


Analyzing cluster size distribution in our entity matching results...

📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,2486,84.100135
1,3,317,10.723951
2,4,93,3.146143
3,5,28,0.947226
4,6,18,0.608931
5,7,7,0.236806
6,8,3,0.101488
7,9,2,0.067659
8,10,1,0.033829
9,11,1,0.033829


In [37]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/music/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 2956 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [38]:
from PyDI.entitymatching import MaximumBipartiteMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_m2d = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 3722 -> 3722 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 3722 -> 2998 
[INFO ] root - MaximumBipartiteMatching: 3722 -> 2998 correspondences
[INFO ] root - MaximumBipartiteMatching: 6655 -> 5996 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2998 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2998	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/music/cluster_analysis/cluster_size_distribution.csv
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  452
[INFO ] root -   True Negatives:  5271
[INFO ] root -   False Positives: 59
[INFO ] root -   False Negatives: 23
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.986
[INFO ] root

In [39]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2l = clusterer.cluster(correspondences_m2l)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  475
[INFO ] root -   True Negatives:  4649
[INFO ] root -   False Positives: 83
[INFO ] root -   False Negatives: 42
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.976
[INFO ] root -   Precision: 0.851
[INFO ] root -   Recall:    0.919
[INFO ] root -   F1-Score:  0.884
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2497 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2178	|	87.22%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	122	|	4.89%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	80	|	3.20%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	24	|	0.96%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	19	|	0.76%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	11	|	0.44%
[INFO ] PyDI.entitymatching.evalua

## Part 4: Data Fusion

In [68]:
mbrainz["mbrainz_id"] = mbrainz["id"]

# Assign trust scores to datasets
mbrainz.attrs["trust_score"] = 3
discogs.attrs["trust_score"] = 1
lastfm.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2l], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 6,081


## Step 1: Define Fusion Strategy 

In [ ]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, maximum

strategy = DataFusionStrategy('music_fusion_strategy')

strategy.add_attribute_fuser('name', shortest_string)
strategy.add_attribute_fuser('artist', longest_string)
strategy.add_attribute_fuser('release-date', prefer_higher_trust)
strategy.add_attribute_fuser('release-country', prefer_higher_trust)
strategy.add_attribute_fuser('duration', maximum)
strategy.add_attribute_fuser('tracks_track_name', union)
strategy.add_attribute_fuser('label', longest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'shortest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'artist' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-date' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-country' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'duration' using rule 'maximum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'tracks_track_name' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'label' using rule 'longest_string'


## Step 2: Run Fusion

In [70]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[mbrainz, discogs, lastfm],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/music/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'music_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 10481 of 10481 unique IDs
[INFO ] PyDI.fusion.engine - Created 31174 record groups from 6081 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 31174 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	2719	|	8.72%
[INFO ] PyDI.fusion.engine - 		3	|	1681	|	5.39%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     artist: 0.43
[INFO ] PyDI.fusion.engine -     duratio

Fused rows: 4,400


,_id,_fusion_sources,_fusion_source_datasets,tracks_track_position,artist,tracks_track_name,release-date,id,release-country,tracks_track_duration,label,genre,duration,name_longest_token,name,mbrainz_id,_fusion_confidence,_fusion_metadata
0,lastFM_67929,"[lastFM_67929, mbrainz_4340, discogs_18531]","[lastfm, musicbrainz, discogs]","[1, 2, 3, 4]",Storm Of Damnation,"[Damage, Guillotine, Life's Liquid, New Song]",1987-01-01,lastFM_67929,United States,"[245, 225, 245, 320]",109 Records,Rock,1035.0,BROKEN,BROKEN,mbrainz_4340,0.687500,{'tracks_track_position_rule': 'first_non_null...
1,mbrainz_134,"[mbrainz_134, lastFM_176, discogs_288]","[musicbrainz, lastfm, discogs]","[1, 2, 3, 4]","Gordon, Alex","[Small Craft Warnings (Alex Gordon Land Mix), ...",2001-01-01,mbrainz_134,United States,"[30, 30, 30, 30]",Electronic 78 Records,Electronic,120.0,Warnings,Small Craft Warnings Remixes,mbrainz_134,0.721065,{'tracks_track_position_rule': 'first_non_null...
2,lastFM_20983,"[lastFM_20983, discogs_44313, mbrainz_9848]","[lastfm, discogs, musicbrainz]","[1, 2, 4, 5, 6, 8, 9, 10]","Blohm, Jan","[9Mm Blues, 9mm Blues, 9mm blues, Aan Elize, A...",2004-01-01,lastFM_20983,South Africa,"[161, 173, 241, 299, 174, 244, 178, 157]",Hathor Records,Blues|Rock,2102.0,Confessions,Melkstraat Confessions,mbrainz_9848,0.670833,{'tracks_track_position_rule': 'first_non_null...
3,discogs_33970,"[discogs_33970, mbrainz_7574, lastFM_16166]","[discogs, musicbrainz, lastfm]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]",Mourning Cloak,"[Bright, Butterfly Sleeping Wind, Chrysalis Re...",1996-01-01,discogs_33970,United States,"[579, 77, 133, 144, 145, 355, 156, 49, 416, 25...",Mindfield Records,"Rock|Folk, World, & Country",3573.0,Dreams,In Dreams You See,mbrainz_7574,0.666667,{'tracks_track_position_rule': 'first_non_null...
4,lastFM_77295,"[lastFM_77295, mbrainz_30996]","[lastfm, musicbrainz]","[1, 2, 4, 6, 7, 8, 10, 11, 14]","Garcia-Fons, Renaud","[Al CamarÃ³n, Aljamiado, Bari, Berimbass, Cami...",2013-11-19,lastFM_77295,France,"[114, 305, 368, 373, 326, 357, 385, 178, 353]",NaN,NaN,4115.0,Beyond,Beyond Double Bass,mbrainz_30996,0.669976,{'tracks_track_position_rule': 'first_non_null...


## Step 3: Evaluate Data Fusion

In [71]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("artist", tokenized_match)
strategy.add_evaluation_function("duration", numeric_tolerance_match, tolerance=10) # 10 seconds tolerance
strategy.add_evaluation_function("release-date", year_only_match)
strategy.add_evaluation_function("release-country", tokenized_match)
strategy.add_evaluation_function("label", tokenized_match)
strategy.add_evaluation_function("tracks_track_name", set_equality_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'artist'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'duration' with params {'tolerance': 10}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-date'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'label'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'tracks_track_name'


In [72]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
# we do not evaluate individual track durations or track position
fusion_test_set = fusion_test_set.drop(columns=["tracks_track_duration"])
fusion_test_set = fusion_test_set.drop(columns=["tracks_track_position"])

# normalize country names in test set reusing mapping from discogs dataset
translator = SchemaTranslator()
spec.set_column("release-country", country_format="name")
mapping = discogs_mapping.copy()
mapping["source_dataset"] = "fusion_test_set"
mapping["source_column"] = mapping["target_column"] # columns already have the correct name
fusion_test_set = translator.translate(
    fusion_test_set, mapping,
    normalize=spec, on_failure="keep"
)

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='mbrainz_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[WARNING] root - Column 'genre' not found in dataset 'fusion_test_set'
[WARNING] root - Column 'tracks_track_position' not found in dataset 'fusion_test_set'
[WARNING] root - Column 'tracks_track_duration' not found in dataset 'fusion_test_set'
[INFO ] root - Translating 7 columns for 'fusion_test_set'
[WARNING] PyDI.normalization.transform - Column 'genre' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'tracks_track_duration' not found in DataFrame
[WARNING] PyDI.normalization.transform - Column 'tracks_track_position' not found in DataFrame
[INFO ] root - Normalization complete: 46 values transformed, 22 values failed
[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/music/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.838 overall accuracy (129/154)
[INFO

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.838
  macro_accuracy: 0.842
  num_evaluated_records: 23
  num_evaluated_attributes: 7
  total_evaluations: 154
  total_correct: 129
  artist_accuracy: 0.870
  artist_count: 23
  release-date_accuracy: 1.000
  release-date_count: 23
  tracks_track_name_accuracy: 0.174
  tracks_track_name_count: 23
  release-country_accuracy: 1.000
  release-country_count: 23
  label_accuracy: 0.938
  label_count: 16
  duration_accuracy: 1.000
  duration_count: 23
  name_accuracy: 0.913
  name_count: 23

Overall Accuracy: 83.8%
